# Master's Thesis Computational System: YouTube Advertising & Influence Knowledge Graph
**Author:** Aliza Hamid  
**Institution:** Universidad Carlos III de Madrid (UC3M) — Master in Big Data Analytics  
**Research Topic:** *Advertising and Influence Analysis via LLM-Generated Graphs from YouTube*  

---
### Overview of the End-to-End Pipeline
This interactive notebook demonstrates the complete computational pipeline developed for the Master's Thesis:
1. **Data Ingestion:** Harvesting YouTube video metadata, engagement statistics, and ASR transcripts via the YouTube Data API v3 and `youtube-transcript-api`.
2. **LLM Knowledge Graph Extraction:** Zero-shot entity and relation extraction powered by Google Gemini 2.5 Flash with structured Pydantic schemas.
3. **Network Science & Modeling:** Constructing a Directed Bipartite Graph $G = (V_C \cup V_B, E)$ with view-weighted edges.
4. **Topological Centrality Analysis:** Calculating Weighted Degree, Eigenvector, and Betweenness Centralities.
5. **Community Detection:** Unsupervised partitioning into industrial sub-niches using Louvain modularity optimization ($Q$).
6. **Information Diffusion:** Simulating promotional cascade propagation using the Independent Cascade Model (ICM).
7. **Interactive Visualization:** Rendering dynamic HTML5 network maps using PyVis / Vis.js.

In [ ]:
# 1. SETUP ENVIRONMENT AND IMPORTS
import os
import sys
import json
import time
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Load API Keys from .env
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

print("Gemini API Configured:", bool(GEMINI_API_KEY))
print("YouTube API Configured:", bool(YOUTUBE_API_KEY))

# Import Modular Thesis Package
from src.config import (
    SCALE_DIVERSE_CHANNELS,
    DEFAULT_GRAPH_DATA_PATH,
    PILOT_GRAPH_DATA_PATH,
    PROCESSED_GRAPH_DATA_PATH
)
from src.data_extraction import YouTubeDataExtractor
from src.kg_extraction import KnowledgeGraphExtractor, BrandMention, ExtractionResult
from src.network_analysis import NetworkAnalyzer
from src.kg_visualization import KnowledgeGraphVisualizer

## Phase 1 & 2: Scale-Diverse Creator Ingestion & LLM Extraction
We sample creators across four distinct ecosystem tiers (Mega-Hubs, Mid-Tier Enthusiasts, Deep-Niche Benchmarkers, and Micro-Specialists).

In [ ]:
# Display the Scale-Diverse Creator Catalog
for ch in SCALE_DIVERSE_CHANNELS:
    print(f"[{ch['tier']}] {ch['name']} ({ch['subscribers']} subs) -> {ch['niche']}")

In [ ]:
# Initialize Network Science Engine and Load Master Graph
analyzer = NetworkAnalyzer()
print(f"Nodes: {analyzer.G.number_of_nodes()} ({len(analyzer.creators)} Creators, {len(analyzer.brands)} Brands)")
print(f"Edges: {analyzer.G.number_of_edges()}")

## Phase 3: Topological Centrality Modeling
We compute Weighted Degree Centrality, Eigenvector Centrality (true influence prestige), and Betweenness Centrality (structural bridge detection).

In [ ]:
# Calculate and Display Network Centrality Metrics
metrics = analyzer.compute_all_metrics()

print("\n--- TOP 5 BRANDS BY EXPOSURE VOLUME (IN-DEGREE VIEWS) ---")
for b in metrics.get("top_brands_by_in_degree", [])[:5]:
    print(f"{b['node']}: {b['score']:,} total views")

print("\n--- TOP 5 INFLUENCERS BY EIGENVECTOR CENTRALITY (PRESTIGE) ---")
for e in metrics.get("top_eigenvector_influencers", [])[:5]:
    print(f"{e['node']} [{e['type']}]: {e['score']:.4f}")

print("\n--- TOP 5 STRUCTURAL BRIDGES (BETWEENNESS CENTRALITY) ---")
for bw in metrics.get("top_betweenness_bridges", [])[:5]:
    print(f"{bw['node']} [{bw['type']}]: {bw['score']:.4f}")

## Phase 4: Community Detection & Sub-Niche Partitioning
Applying Louvain modularity maximization to identify latent sponsorship clusters.

In [ ]:
# Detect Communities
communities = analyzer.detect_communities()
print(f"Total Detected Clusters: {communities.get('num_communities')} (Modularity Q = {communities.get('modularity_score', 0):.4f})")

for c in communities.get("clusters", []):
    print(f"\nCluster {c['community_id']} (Size: {c['size']} entities | {c['num_creators']} creators, {c['num_brands']} brands):")
    print("  Creators:", ", ".join(c['lead_creators']))
    print("  Top Brands:", ", ".join(c['lead_brands']))

## Phase 5: Information Diffusion Simulation (Independent Cascade Model)
Simulating cascade reach comparing a Mega-Hub concentration strategy vs a Distributed Mid-Tier strategy.

In [ ]:
# Run Cascade Simulations
diffusion_results = analyzer.simulate_information_diffusion(propagation_prob=0.35, monte_carlo_trials=50)

for strat, res in diffusion_results.items():
    print(f"{strat}: Mean Reach = {res['mean_final_reach']:.1f} nodes ({res['reach_percentage']:.1f}% of network)")

## Phase 6: Interactive Knowledge Graph Visualization
Rendering and exporting the interactive HTML5 Vis.js graph.

In [ ]:
# Generate Interactive PyVis HTML Map & Publication Figures
visualizer = KnowledgeGraphVisualizer(analyzer)
html_path = visualizer.generate_interactive_html()
figs = visualizer.generate_publication_figures()
print(f"[+] Interactive Visualization generated at: {html_path}")
print(f"[+] Figures generated: {list(figs.keys())}")